# TETOUAN-POWER-ML
## 1. Problema
Predecir Zona 1 a 60 minutos con información causal.

In [ ]:
from pathlib import Path
import os, subprocess, sys
if not Path('pyproject.toml').exists():
    if not Path('ML').exists():
        subprocess.run(['git','clone','--branch','feature/tetouan-power-ml','https://github.com/albarracinelvys-droid/ML.git'],check=True)
    os.chdir('ML')
subprocess.run([sys.executable,'-m','pip','install','-e','.','xgboost','lightgbm','catboost','optuna'],check=True)

## 2. Dataset
UCI 849, CC BY 4.0. No modificar el archivo original.

In [ ]:
from tetouan.data import download, read
path=download('data/raw/tetouan.csv')
frame=read(path)
print(frame.shape, frame.index.min(), frame.index.max())

## 3. EDA e ingeniería
Exploración inicial solo del tramo de entrenamiento.

In [ ]:
display(frame.iloc[:int(len(frame)*.7)].describe())

### Variables ampliadas y redundancia
59 predictores causales. Comparar correlaciones, distribuciones y asimetría solamente en entrenamiento. Consumo histórico disponible en t no es la etiqueta futura y(t+60). El test ya observado en la versión inicial debe considerarse exploratorio en esta ampliación.

In [ ]:
from tetouan.analyze import analyze
analyze(path, 'artifacts/expanded')
import pandas as pd
display(pd.read_csv('artifacts/expanded/feature_statistics.csv'))
display(pd.read_csv('artifacts/expanded/redundant_pairs.csv').head(20))

## 4. Modelado clásico y RealMLP
Presupuesto inicial de 16 épocas, CPU.

In [ ]:
from tetouan.train import run
run(path,'artifacts',epochs=16)

## 5. Evaluación
Selección en validación; test no interviene en selección.

In [ ]:
import pandas as pd
display(pd.read_csv('artifacts/validation.csv'))
display(pd.read_csv('artifacts/test.csv'))

## 6. Interpretabilidad
Importancias por permutación; asociaciones, no efectos causales.

In [ ]:
display(pd.read_csv('artifacts/importance.csv'))

## 7. Mini-despliegue y comunicación
Ejecutar localmente `python -m streamlit run app.py`. En Colab se valida inferencia sin exponer un servidor público.

In [ ]:
import joblib
from tetouan.inference import predict
print(predict(joblib.load('artifacts/model.joblib'),frame))